# Multivariable Cox Analysis

Show that the model's risk score is an **independent prognostic factor** after adjusting for standard clinical covariates (age, stage, grade).

Comparisons:
- **Clinical only**: `time ~ age + stage + grade` → C-index
- **Model only**: `time ~ risk_score` → C-index
- **Combined**: `time ~ age + stage + grade + risk_score` → C-index

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
from pathlib import Path
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 11,
    'axes.linewidth': 0.8,
})

In [ ]:
# === Configure runs ===
RUNS = {
    'Image Only':     '../runs/mmp20_image_only_cos20_es_20260303_130951',
    'Ours':           '../runs/mmp20_sq_cos20_knn64_es_20260303_132721',
}

CLINICAL_TSV = Path('../data/clinical.tsv')

# Toggle: include stage as a covariate? (False = use age+sex+grade only)
USE_STAGE = False

In [ ]:
# === Load patient-level predictions from all folds ===
def load_predictions(run_dir):
    run_dir = Path(run_dir)
    all_patients = []
    for ff in sorted(run_dir.glob('fold_*_result.json')):
        with open(ff) as f:
            data = json.load(f)
        for p in data.get('test_patient_results', []):
            p['fold'] = data['fold']
            all_patients.append(p)
    return pd.DataFrame(all_patients)

predictions = {}
for name, path in RUNS.items():
    predictions[name] = load_predictions(path)
    print(f'{name}: {len(predictions[name])} patients across {predictions[name]["fold"].nunique()} folds')

In [ ]:
# === Load clinical covariates ===
clin = pd.read_csv(CLINICAL_TSV, sep='\t', low_memory=False)

# Keep relevant columns
clin = clin[[
    'cases.submitter_id', 'project.project_id',
    'demographic.age_at_index', 'demographic.gender',
    'diagnoses.ajcc_pathologic_stage', 'diagnoses.tumor_grade',
]].rename(columns={
    'cases.submitter_id': 'patient_id',
    'project.project_id': 'cancer_type',
    'demographic.age_at_index': 'age',
    'demographic.gender': 'gender',
    'diagnoses.ajcc_pathologic_stage': 'stage',
    'diagnoses.tumor_grade': 'grade',
})

# Deduplicate: keep first non-null entry per patient
# Sort so non-null stage/grade come first
clin['stage_available'] = clin['stage'].notna() & (clin['stage'] != "'--")
clin['grade_available'] = clin['grade'].notna() & (clin['grade'] != "'--")
clin = clin.sort_values(['patient_id', 'stage_available', 'grade_available'], ascending=[True, False, False])
clin = clin.drop_duplicates(subset='patient_id', keep='first')
clin = clin.drop(columns=['stage_available', 'grade_available'])

# Clean up sentinel values
clin['stage'] = clin['stage'].replace("'--", np.nan)
clin['grade'] = clin['grade'].replace("'--", np.nan)

print(f'Clinical data: {len(clin)} unique patients')
print(f'Age available: {clin["age"].notna().sum()}')
print(f'Stage available: {clin["stage"].notna().sum()}')
print(f'Grade available: {clin["grade"].notna().sum()}')
print(f'\nStage values: {clin["stage"].value_counts().head(10).to_dict()}')
print(f'\nGrade values: {clin["grade"].value_counts().head(10).to_dict()}')

In [ ]:
# === Encode clinical variables ===
def encode_stage(s):
    """Map AJCC stage to ordinal: I=1, II=2, III=3, IV=4."""
    if pd.isna(s):
        return np.nan
    s = str(s).upper().strip()
    if 'IV' in s: return 4
    if 'III' in s: return 3
    if 'II' in s: return 2
    if 'I' in s: return 1
    return np.nan

def encode_grade(g):
    """Map tumor grade to ordinal."""
    if pd.isna(g):
        return np.nan
    g = str(g).upper().strip()
    # Handle G1/G2/G3/G4
    if 'G4' in g or 'HIGH GRADE' in g: return 4
    if 'G3' in g: return 3
    if 'G2' in g: return 2
    if 'G1' in g or 'GX' in g or 'LOW GRADE' in g: return 1
    return np.nan

clin['stage_ord'] = clin['stage'].apply(encode_stage)
clin['grade_ord'] = clin['grade'].apply(encode_grade)
clin['age'] = pd.to_numeric(clin['age'], errors='coerce')
clin['female'] = (clin['gender'] == 'female').astype(float)

print(f'Stage ordinal: {clin["stage_ord"].value_counts().sort_index().to_dict()}')
print(f'Grade ordinal: {clin["grade_ord"].value_counts().sort_index().to_dict()}')
print(f'Age: mean={clin["age"].mean():.1f}, std={clin["age"].std():.1f}')
print(f'Female: {clin["female"].mean():.1%}')

In [ ]:
# === Merge predictions with clinical data (per model, per fold) ===
def build_cox_df(pred_df, clin_df):
    """Merge predictions with clinical covariates."""
    merged = pred_df.merge(clin_df[['patient_id', 'age', 'female', 'stage_ord', 'grade_ord']],
                          on='patient_id', how='left')
    print(f'  Merged: {len(merged)} patients')
    print(f'  Missing age: {merged["age"].isna().sum()}')
    print(f'  Missing stage: {merged["stage_ord"].isna().sum()}')
    print(f'  Missing grade: {merged["grade_ord"].isna().sum()}')
    return merged

merged = {}
for name in RUNS:
    print(f'\n{name}:')
    merged[name] = build_cox_df(predictions[name], clin)

In [ ]:
# === Per-fold multivariable Cox analysis ===
# For each fold, fit Cox models on the TEST set predictions
# (we're not training new models — just testing if risk_score adds
# independent prognostic value beyond clinical covariates)

CLINICAL_VARS = ['age', 'female', 'grade_ord']
if USE_STAGE:
    CLINICAL_VARS.append('stage_ord')
print(f'Clinical covariates: {CLINICAL_VARS}')

def fit_cox_models(df, model_name):
    """Fit 3 Cox models per fold and return C-indices."""
    folds = sorted(df['fold'].unique())
    results = []

    for fold in folds:
        fold_df = df[df['fold'] == fold].copy()

        # Drop patients with missing covariates for fair comparison
        complete = fold_df.dropna(subset=CLINICAL_VARS + ['risk'])
        n_dropped = len(fold_df) - len(complete)

        if len(complete) < 20:
            print(f'  Fold {fold}: too few complete cases ({len(complete)}), skipping')
            continue

        # Standardize risk score for comparable HRs
        complete['risk_z'] = (complete['risk'] - complete['risk'].mean()) / (complete['risk'].std() + 1e-8)
        # Standardize age
        complete['age_z'] = (complete['age'] - complete['age'].mean()) / (complete['age'].std() + 1e-8)

        # Build covariate list with standardized age
        cov_cols = ['age_z' if v == 'age' else v for v in CLINICAL_VARS]

        fold_result = {'fold': fold, 'n': len(complete), 'n_dropped': n_dropped}

        # Model 1: Clinical only
        try:
            cph_clin = CoxPHFitter()
            cph_clin.fit(complete[['time', 'event'] + cov_cols],
                        duration_col='time', event_col='event')
            fold_result['c_clinical'] = cph_clin.concordance_index_
        except Exception as e:
            fold_result['c_clinical'] = np.nan

        # Model 2: Model risk score only
        try:
            cph_model = CoxPHFitter()
            cph_model.fit(complete[['time', 'event', 'risk_z']],
                        duration_col='time', event_col='event')
            fold_result['c_model'] = cph_model.concordance_index_
        except Exception as e:
            fold_result['c_model'] = np.nan

        # Model 3: Clinical + model risk score
        try:
            cph_combined = CoxPHFitter()
            cph_combined.fit(complete[['time', 'event'] + cov_cols + ['risk_z']],
                           duration_col='time', event_col='event')
            fold_result['c_combined'] = cph_combined.concordance_index_
            fold_result['summary'] = cph_combined.summary
        except Exception as e:
            fold_result['c_combined'] = np.nan

        results.append(fold_result)

    return results

cox_results = {}
for name in RUNS:
    print(f'\n=== {name} ===')
    cox_results[name] = fit_cox_models(merged[name], name)

## Molecular Covariate Extension

Extend the multivariable Cox analysis by adding established molecular biomarkers as additional covariates for cancer types where they are known to be prognostic:

- **LGG**: IDH mutation + 1p/19q codeletion
- **BRCA**: ER/PR/HER2 status (PAM50 subtype)
- **CRC** (COAD+READ): MSI status
- **UCEC**: Molecular subtype (POLE/MSI/CN_LOW/CN_HIGH)
- **HNSC**: HPV status
- **STAD**: Molecular subtype (CIN/MSI/GS/EBV)
- **LUAD**: EGFR/KRAS mutation status

**Goal**: Show that SPARC-Risk remains independently prognostic even after adjusting for these strong molecular prognostic factors.

In [ ]:
# === Load molecular covariates ===
mol = pd.read_csv('../data/molecular_covariates.csv')

# Merge COAD + READ into CRC in the molecular data
mol.loc[mol['cancer_type'].isin(['TCGA-COAD', 'TCGA-READ']), 'cancer_type'] = 'TCGA-CRC'

print(f"Molecular covariates: {len(mol)} patients")
print(f"\nPer cancer type:")
print(mol.groupby('cancer_type').size().to_string())

# Show coverage of each biomarker
print("\n\nBiomarker coverage:")
for col in ['idh_status', 'codel_1p19q', 'pam50_subtype', 'er_status', 'pr_status', 
            'her2_status', 'msi_status', 'molecular_subtype', 'hpv_status', 'egfr_mutated', 'kras_mutated']:
    n = mol[col].notna().sum()
    if n > 0:
        print(f"  {col:25s}: {n:4d} patients — {dict(mol[col].dropna().value_counts())}")

In [ ]:
# === Molecular-extended multivariable Cox ===
# For each cancer, compare:
#   1. Clinical only (age + sex + grade)
#   2. Clinical + molecular biomarker
#   3. Clinical + molecular + SPARC-Risk
# Show that risk score remains independently prognostic


MOLECULAR_CONFIGS = {
    'LGG': {
        'mol_cancer': 'TCGA-LGG',
        'columns': ['idh_status', 'codel_1p19q'],
        'encode': lambda df: df.assign(
            idh_mutant=(df['idh_status'] == 'mutant').astype(float),
            codel=(df['codel_1p19q'] == 'codel').astype(float),
        ),
        'covs': ['idh_mutant', 'codel'],
        'label': 'IDH + 1p/19q',
    },
    'BRCA': {
        'mol_cancer': 'TCGA-BRCA',
        'columns': ['er_status', 'pr_status', 'her2_status'],
        'encode': lambda df: df.assign(
            er_pos=(df['er_status'] == 'Positive').astype(float),
            pr_pos=(df['pr_status'] == 'Positive').astype(float),
            her2_pos=(df['her2_status'] == 'Positive').astype(float),
        ),
        'covs': ['er_pos', 'pr_pos', 'her2_pos'],
        'label': 'ER/PR/HER2',
    },
    'CRC': {
        'mol_cancer': 'TCGA-CRC',
        'columns': ['msi_status'],
        'encode': lambda df: df.assign(
            msi=(df['msi_status'] == 'MSI').astype(float),
        ),
        'covs': ['msi'],
        'label': 'MSI',
    },
    'UCEC': {
        'mol_cancer': 'TCGA-UCEC',
        'columns': ['molecular_subtype'],
        'encode': lambda df: df.assign(
            # CN_HIGH is the key prognostic factor (worst prognosis)
            # POLE has only 1 event (quasi-separation) → not usable as separate covariate
            # Encoding: CN_HIGH vs rest (CN_LOW + MSI + POLE)
            is_CN_HIGH=(df['molecular_subtype'] == 'CN_HIGH').astype(float),
            is_MSI=(df['molecular_subtype'] == 'MSI').astype(float),
        ),
        'covs': ['is_CN_HIGH', 'is_MSI'],
        'label': 'Mol. subtype',
    },
    'HNSC': {
        'mol_cancer': 'TCGA-HNSC',
        'columns': ['hpv_status'],
        'encode': lambda df: df.assign(
            hpv_pos=(df['hpv_status'] == 'HPV+').astype(float),
        ),
        'covs': ['hpv_pos'],
        'label': 'HPV',
    },
    'LUAD': {
        'mol_cancer': 'TCGA-LUAD',
        'columns': ['egfr_mutated', 'kras_mutated'],
        'encode': lambda df: df.assign(
            egfr_mut=df['egfr_mutated'].fillna(False).astype(float),
            kras_mut=df['kras_mutated'].fillna(False).astype(float),
        ),
        'covs': ['egfr_mut', 'kras_mut'],
        'label': 'EGFR/KRAS',
    },
}

# Use "Ours" predictions; merge COAD+READ into CRC
pred = predictions['Ours'].copy()
pred.loc[pred['cancer_type'].isin(['TCGA-COAD', 'TCGA-READ']), 'cancer_type'] = 'TCGA-CRC'

results_rows = []

for cancer_short, cfg in MOLECULAR_CONFIGS.items():
    cancer_full = cfg['mol_cancer']
    
    # Get predictions for this cancer
    cancer_pred = pred[pred['cancer_type'] == cancer_full].copy()
    if len(cancer_pred) == 0:
        print(f"\n{cancer_short}: No predictions found")
        continue
    
    # Merge clinical
    clinical_cols = ['patient_id', 'age', 'female', 'grade_ord']
    cancer_df = cancer_pred.merge(clin[clinical_cols], on='patient_id', how='left')
    
    # Merge molecular
    mol_cols = ['patient_id'] + cfg['columns']
    cancer_mol = mol[mol['cancer_type'] == cancer_full][mol_cols].drop_duplicates('patient_id')
    cancer_df = cancer_df.merge(cancer_mol, on='patient_id', how='left')
    
    # Encode molecular covariates
    cancer_df = cfg['encode'](cancer_df)
    
    # Build covariate lists
    base_covs = ['age']
    if cancer_full not in SEX_SPECIFIC:
        base_covs.append('female')
    # Only include grade if enough non-null values AND enough unique values
    n_grade = cancer_df['grade_ord'].notna().sum()
    n_grade_unique = cancer_df['grade_ord'].dropna().nunique()
    if n_grade >= 10 and n_grade_unique >= 2:
        base_covs.append('grade_ord')
    
    mol_covs = cfg['covs']
    
    # Drop rows with missing values
    all_covs = base_covs + mol_covs + ['risk']
    sub = cancer_df.dropna(subset=all_covs + ['time', 'event']).copy()
    n_events = int(sub['event'].sum())
    n_total = len(sub)
    
    # Also get count without molecular (for comparison)
    sub_clin_only = cancer_df.dropna(subset=base_covs + ['risk', 'time', 'event']).copy()
    n_clin_only = len(sub_clin_only)
    
    print(f"\n{'='*60}")
    print(f"{cancer_short} ({cfg['label']}): n={n_total} (events={n_events}), "
          f"n_clinical_only={n_clin_only}")
    print(f"  Clinical covariates: {base_covs}")
    print(f"  Molecular covariates: {mol_covs}")
    for mc in mol_covs:
        print(f"    {mc}: {sub[mc].value_counts().to_dict()}")
    
    if n_events < 10:
        print(f"  SKIPPED: too few events ({n_events})")
        continue
    
    # Standardize continuous variables
    sub['risk_z'] = (sub['risk'] - sub['risk'].mean()) / (sub['risk'].std() + 1e-8)
    sub['age_z'] = (sub['age'] - sub['age'].mean()) / (sub['age'].std() + 1e-8)
    cov_cols_z = ['age_z' if v == 'age' else v for v in base_covs]
    
    row = {
        'cancer': cancer_short,
        'molecular': cfg['label'],
        'n': n_total,
        'events': n_events,
    }
    
    # Use higher penalizer for robustness (helps with sparse molecular subtypes)
    PEN = 0.1
    
    # Model 1: Clinical only
    try:
        cph1 = CoxPHFitter(penalizer=PEN)
        cph1.fit(sub[['time', 'event'] + cov_cols_z], duration_col='time', event_col='event')
        row['c_clinical'] = cph1.concordance_index_
    except Exception as e:
        print(f"  Clinical-only failed: {e}")
        row['c_clinical'] = np.nan
    
    # Model 2: Clinical + molecular
    try:
        cph2 = CoxPHFitter(penalizer=PEN)
        cph2.fit(sub[['time', 'event'] + cov_cols_z + mol_covs], duration_col='time', event_col='event')
        row['c_clin_mol'] = cph2.concordance_index_
    except Exception as e:
        print(f"  Clinical+molecular failed: {e}")
        row['c_clin_mol'] = np.nan
    
    # Model 3: Clinical + molecular + risk score
    try:
        all_covs_z = cov_cols_z + mol_covs + ['risk_z']
        cph3 = CoxPHFitter(penalizer=PEN)
        cph3.fit(sub[['time', 'event'] + all_covs_z], duration_col='time', event_col='event')
        row['c_clin_mol_risk'] = cph3.concordance_index_
        
        # Extract HR and p-value for risk_z
        row['hr_risk'] = np.exp(cph3.params_['risk_z'])
        ci = cph3.confidence_intervals_.loc['risk_z']
        row['hr_lo'] = np.exp(ci.iloc[0])
        row['hr_hi'] = np.exp(ci.iloc[1])
        row['p_risk'] = cph3.summary.loc['risk_z', 'p']
        
        print(f"  C-index: clinical={row['c_clinical']:.3f} → +mol={row['c_clin_mol']:.3f} → +risk={row['c_clin_mol_risk']:.3f}")
        print(f"  Risk HR={row['hr_risk']:.2f} [{row['hr_lo']:.2f}–{row['hr_hi']:.2f}], p={row['p_risk']:.4f}"
              f" {'***' if row['p_risk'] < 0.001 else '**' if row['p_risk'] < 0.01 else '*' if row['p_risk'] < 0.05 else 'ns'}")
        
        # Print all covariates from the full model
        print(f"\n  Full model summary (clinical + {cfg['label']} + risk):")
        for var in cph3.summary.index:
            hr = np.exp(cph3.params_[var])
            p = cph3.summary.loc[var, 'p']
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            print(f"    {var:20s}: HR={hr:.3f}, p={p:.4f} {sig}")
    except Exception as e:
        print(f"  Full model failed: {e}")
        row['c_clin_mol_risk'] = np.nan
        row['hr_risk'] = np.nan
        row['p_risk'] = np.nan
    
    results_rows.append(row)

mol_results = pd.DataFrame(results_rows)
print(f"\n\n{'='*60}")
print("SUMMARY: SPARC-Risk independence from molecular biomarkers")
print(f"{'='*60}")
sig_count = (mol_results['p_risk'] < 0.05).sum()
total = mol_results['p_risk'].notna().sum()
print(f"\nSPARC-Risk independently prognostic in {sig_count}/{total} cancer types")
print(f"(after adjusting for age, sex, grade, AND molecular biomarkers)\n")

In [ ]:
# === Bar chart: C-index progression (Clinical → +Molecular → +SPARC-Risk) ===
valid = mol_results.dropna(subset=['c_clinical', 'c_clin_mol', 'c_clin_mol_risk']).copy()
valid = valid.sort_values('c_clin_mol_risk', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(3, len(valid) * 0.8)))

y = np.arange(len(valid))
height = 0.25

bars1 = ax.barh(y - height, valid['c_clinical'], height, color='#D1D1D1', label='Clinical (age+sex+grade)')
bars2 = ax.barh(y, valid['c_clin_mol'], height, color='#E8A855', label='+ Molecular biomarker')
bars3 = ax.barh(y + height, valid['c_clin_mol_risk'], height, color='#2B6B8A', label='+ SPARC-Risk')

# Add significance stars
for i, (_, row) in enumerate(valid.iterrows()):
    sig = '***' if row['p_risk'] < 0.001 else '**' if row['p_risk'] < 0.01 else '*' if row['p_risk'] < 0.05 else 'ns'
    ax.text(row['c_clin_mol_risk'] + 0.005, i + height, sig, va='center', fontsize=9, fontweight='bold',
            color='#2B6B8A' if sig != 'ns' else '#999')

ax.set_yticks(y)
ax.set_yticklabels([f"{r['cancer']} ({r['molecular']})" for _, r in valid.iterrows()], fontsize=10)
ax.set_xlabel('C-index', fontsize=12)
ax.set_title('C-index: Clinical → +Molecular → +SPARC-Risk', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0.45, max(valid['c_clin_mol_risk'].max(), valid['c_clin_mol'].max()) + 0.08)
ax.axvline(x=0.5, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('../figs/multivariable_cox_progression.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary: how much does SPARC-Risk add beyond molecular?
delta = valid['c_clin_mol_risk'] - valid['c_clin_mol']
print(f"\nC-index improvement from adding SPARC-Risk on top of clinical + molecular:")
for _, row in valid.iterrows():
    d = row['c_clin_mol_risk'] - row['c_clin_mol']
    print(f"  {row['cancer']:8s}: {d:+.3f}")
print(f"\n  Mean improvement: {delta.mean():+.3f}")